# Business Understanding
Het lectoraat energietransitie is bezig om huizen duurzamer te maken. Gezien ongeveer 1/3e deel van de co2 uitstoot wordt veroorzaakt ligt daar de focus voor dit project. Om grote stappen te maken ligt de focus dan ook op de al bestaande huizen. De vraag hierin is dan ook. Hoe kun je huizen verwarmen zonder of door zo minimaal mogelijk co2 uit te stoten. 

Hiervoor heeft het lectoraat al verschillende metingen gedaan om er achter te komen hoeveel warmte verliest een huis dan daadwerkelijk, deze metingen zijn vooral gedaan door eigen apparaten te maken en de data te meten. Ook hebben ze in samenwerking met Enelogic en huiseigenaren de data vanuit de boilers kunnen verkrijgen.

## Hoe wordt een huis verwarmt
Een huis wordt grotendeels verwarmd door de kachels die in huis staan, maar de zon levert ook warmte. dat is waar ik voornamelijk mijn focus op leg. In dit notebook ga ik mij vooral focussen om de warmte van de zon te berekenen die je binnenkrijgt in je huis. 

## Waar ga ik mee bezig
Hiervoor heb ik de volgende berekening gekregen: solar irradiation = GH"I [W/m2]"× Δt [s] × Asol [m2]. de GHI kan ik verkrijgen uit de dataset van de KNMI. dit wordt gemeten in de volgende standaard: Jules per cm2 gedurende het uur, Δt is hoelang de zon schijnt, Asol in m2 is het raamoppervlak.

# Data Understanding en Preperation

hier importeer ik alle bestanden die ik nodig heb om te kunnen kijken wat er mogelijk is. ook is dit het stukje waar ik de data voorbereid om mee te werk te gaan. verderop zorg ik er voor dat de data gecombineerd wordt tot 1 dataset. dit doe ik zodat ik bepaalde stukken data tegenover elkaar kan zetten.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
needforheat_file_path='../needforheat_export_student_EM.parquet'

In [ ]:
%%time
# Attempt to read the Parquet file needforheat
try:
    df_nfh= pd.read_parquet(
        needforheat_file_path,
        engine='pyarrow',
        #use_nullable_dtypes=True #Outcommented because of futureWarning
        dtype_backend='numpy_nullable'
    )
    print("File was successfully read without specifying compression codec.")
except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
df_nfh = df_nfh.pivot_table(index=['id','timestamp', 'source_category', 'source_type'],
                            columns='property', values='value').reset_index()

In [ ]:
df_nfh = df_nfh.groupby('timestamp').agg({
    'id': 'first',  # Assuming 'id' is the same for the same timestamp
    'g_use_cum__m3': 'sum'  # Summing values, modify as needed
}).reset_index()

In [ ]:
df_nfh['id'].unique().tolist()

In [ ]:
Remeha_dataset = '..//remeha_export_student.parquet'

In [ ]:

%%time
# Attempt to read the Parquet file needforheat
try:
    df_rmh = pd.read_parquet(
        Remeha_dataset,
        engine='pyarrow',
        dtype_backend='numpy_nullable'
    )
    print("File was successfully read without specifying compression codec.")
except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
df_rmh.reset_index(inplace=True)

In [ ]:
df_rmh

In [ ]:
#df_rmh = df_rmh.pivot_table(index=['id', 'timestamp', 'source_category', 'source_type'],columns='property', values='value').reset_index()
#eerst maakte ik gebruik van een pivot table, dit duurde best lang om te wachten tot het klaar was. Door de tip van Erik heeft hij mij er op gewezen dat het sneller kan. Ik heb het hieronder nog een aantal keer gebruikt.

df_rmh['g_use_ch_inf_cum__kWh'] = df_rmh[df_rmh['property'] == 'g_use_ch_inf_cum__kWh']['value']
df_rmh['g_use_dhw_inf_cum__kWh'] = df_rmh[df_rmh['property'] == 'g_use_dhw_inf_cum__kWh']['value']
df_rmh['temp_in__degC'] = df_rmh[df_rmh['property'] == 'temp_in__degC']['value']
df_rmh.drop(columns=['property', 'value'], inplace=True)

df_rmh

In [ ]:
df_rmh.info()

In [ ]:
df_rmh = df_rmh.groupby('timestamp').agg({
    'id': 'first',  # Assuming 'id' is the same for the same timestamp
    'g_use_ch_inf_cum__kWh': 'sum',  # Summing values, modify as needed
    'temp_in__degC': 'mean' # Taking the mean, modify as needed
}).reset_index()

df_rmh

Eerst maakte ik gebruik van de import die erik gemaakt heeft om de data voor het weer elke keer opnieuw op te halen, omdat de data in het verleden ligt was het gemakkelijker om het op te halen uit de historic weather en vervolgens om te zetten in een csv-bestand. Dit heeft er voor gezorgd dat het een stuk sneller gaat dan voorheen.

In [ ]:
df_wtr = pd.read_csv('weather.csv')

df_wtr

In [ ]:
df_wtr.info()

In [ ]:
df_floor_area_m2 = pd.read_csv('floor_area_m2.csv', sep=';')

df_floor_area_m2

In [ ]:
df_floor_area_m2.info()

In [ ]:
df_floor_area_m2.rename(columns={'students_data_science__id': 'Id'}, inplace=True)

In [ ]:
df_floor_area_m2

In [ ]:
sane_measurements = '..//reducedheatcarb_sane_measurements_student.parquet'

In [ ]:

%%time
# Attempt to read the Parquet file needforheat
try:
    df_set_temp = pd.read_parquet(
        sane_measurements,
        engine='pyarrow',
        #use_nullable_dtypes=True #Outcommented because of futureWarning
        dtype_backend='numpy_nullable'
    )
    print("File was successfully read without specifying compression codec.")
except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
df_set_temp.reset_index(inplace=True)
#df_set_temp.set_index('timestamp', inplace=True)
df_set_temp

In [ ]:
df_set_temp['temp_set__degC'] = df_set_temp[df_set_temp['property'] == 'temp_set__degC']['value']
df_set_temp.drop(columns=['property', 'value'], inplace=True)

In [ ]:
df_set_temp

In [ ]:
df_set_temp.set_index('timestamp',inplace=True)

In [ ]:
agg_dict = {
    'id':'last',        
    'temp_set__degC': 'mean' # Average temperature of the hour

}

# Resample the DataFrame to hourly frequency and apply the aggregations
df_set_temp = df_set_temp.resample('H').agg(agg_dict)

# Reset the index if you want the timestamp as a column again
df_set_temp.reset_index(inplace=True)

# Display the resampled DataFrame
df_set_temp

In [ ]:
df_rmh

In [ ]:
#df_rmh_filtered = df_rmh
#verander de timestamp naam van 'Unnamed: 0' naar 'timestamp'
df_wtr['timestamp'] = pd.to_datetime(df_wtr['Unnamed: 0'])  # Gebruik de juiste kolom voor datetime

In [ ]:
# Remove the 'unnamed: 0' column from df_wtr
df_wtr = df_wtr.drop(columns=['Unnamed: 0'])
columns = ['timestamp'] + [col for col in df_wtr if col != 'timestamp']
df_wtr = df_wtr[columns]
df_wtr.rename(columns={'T': 'temp_outside', 'Q': 'GHI'}, inplace=True)
df_wtr

# Definieer de cutoff datetime
cutoff_datetime = pd.Timestamp('2024-03-31 03:00:00+02:00')

# Filter de DataFrame om rijen te behouden waarvan de 'timestamp' kleiner is dan cutoff_datetime
df_wtr = df_wtr[df_wtr['timestamp'] < cutoff_datetime]

df_wtr

In [ ]:
df_hwk1 = pd.read_csv('../Erik/df_day_H__W_k_1.csv')

df_hwk1

In [ ]:
selected_columns = ['Unnamed: 0', 'id', 'timestamp', 'heat_gain_ch__J', 'H__W_K_1']
df_hwk1 = df_hwk1[selected_columns]

In [ ]:
# Specificeer de id om te filteren
specific_id = 559286
#verschillende id's beschikbaar om uit te kiezen
#573579, 538728,590793,522131,528230,545968,557614,504509,526833,579485,562321,581277,542663,565038,528213,538163,579082,565396,559286,504607

# Filter dataframes op het specifieke id
#df_rmh_filtered = df_rmh[df_rmh['id'] == specific_id]
#df_nfh = df_nfh[df_nfh['id'] == specific_id]
#df_wtr = df_wtr[df_wtr['id'] == specific_id]
#df_hwk1 = df_hwk1[df_hwk1['id'] == specific_id]

In [ ]:
# Verwijder dubbele waarden op basis van 'timestamp'
df_rmh = df_rmh.drop_duplicates(subset=['timestamp'])
df_nfh = df_nfh.drop_duplicates(subset=['timestamp'])
df_wtr = df_wtr.drop_duplicates(subset=['timestamp'])
df_hwk1= df_hwk1.drop_duplicates(subset=['timestamp'])
df_set_temp = df_set_temp.drop_duplicates(subset=['timestamp'])

# Converteer de 'timestamp' kolommen naar datetime en verwijder tijdzone-informatie of converteer naar UTC
df_rmh['timestamp'] = pd.to_datetime(df_rmh['timestamp'], utc=True)
df_nfh['timestamp'] = pd.to_datetime(df_nfh['timestamp'], utc=True)
df_wtr['timestamp'] = pd.to_datetime(df_wtr['timestamp'], utc=True)
df_hwk1['timestamp'] = pd.to_datetime(df_hwk1['timestamp'], utc=True)
df_set_temp['timestamp'] = pd.to_datetime(df_set_temp['timestamp'], utc=True)

# Controleer dat 'timestamp' kolommen het juiste datatype hebben
df_rmh['timestamp'] = pd.to_datetime(df_rmh['timestamp'])
df_nfh['timestamp'] = pd.to_datetime(df_nfh['timestamp'])
df_wtr['timestamp'] = pd.to_datetime(df_wtr['timestamp'])
df_hwk1['timestamp'] = pd.to_datetime(df_hwk1['timestamp'])
df_set_temp['timestamp'] = pd.to_datetime(df_set_temp['timestamp'])

# Verwijder tijdzone-informatie als je de tijdzone niet nodig hebt
df_rmh['timestamp'] = df_rmh['timestamp'].dt.tz_convert(None)
df_nfh['timestamp'] = df_nfh['timestamp'].dt.tz_convert(None)
df_wtr['timestamp'] = df_wtr['timestamp'].dt.tz_convert(None)
df_hwk1['timestamp'] = df_hwk1['timestamp'].dt.tz_convert(None)
df_set_temp['timestamp'] = df_set_temp['timestamp'].dt.tz_convert(None)

# Stel 'timestamp' in als index voor elke gefilterde dataframe
df_rmh.set_index('timestamp', inplace=True)
df_nfh.set_index('timestamp', inplace=True)
df_wtr.set_index('timestamp', inplace=True)
df_hwk1.set_index('timestamp', inplace=True)
df_set_temp.set_index('timestamp', inplace=True)

# Combineer de dataframes
df_combined = pd.concat([df_rmh, df_nfh, df_wtr,df_hwk1,df_set_temp], join='outer')

# Verwijder 'Unnamed: 0' kolom als deze bestaat
if 'Unnamed: 0' in df_combined.columns:
    df_combined.drop(columns=['Unnamed: 0'], inplace=True)

# Reset de index om 'timestamp' terug te krijgen als kolom
df_combined.reset_index(inplace=True)

# Stel 'timestamp' in als eerste kolom en als index
df_combined.set_index('timestamp', inplace=True)
df_combined.index.name = 'timestamp'  # Stel de indexnaam opnieuw in (optioneel)

# Print de eerste paar rijen om te controleren
df_combined


In [ ]:
df_combined.info()

In [ ]:
# Bereken de warmtevraag
df_combined['temp_demand__degC'] = df_combined['temp_in__degC'] - df_combined['temp_outside']
# Bereken het verschil in gasverbruik
df_combined['gas_usage_diff'] = df_combined['g_use_cum__m3'].diff().fillna(0)
df_combined

In [ ]:
# Definieer de start- en einddatum
start_date = pd.Timestamp('2024-01-26')
end_date = pd.Timestamp('2024-03-27')

# Filter de dataframe tussen de start- en einddatum
#df_combined= df_combined[(df_combined['timestamp'] >= start_date) & (df_combined['timestamp'] < end_date)]


df_combined

In [ ]:
# Assuming df_combined is already defined and has a datetime index
# Ensure 'timestamp' is the index and in datetime format
#df_combined['timestamp'] = pd.to_datetime(df_combined['timestamp'])
#df_combined.set_index('timestamp', inplace=True)

# Define the aggregation dictionary
agg_dict = {
    'id':'last',
    'g_use_ch_inf_cum__kWh': 'last',   # Last value of the hour
    'temp_in__degC': 'mean',         # Average temperature of the hour
    'temp_outside': 'mean',        # Already per hour, so mean is fine
    'GHI': 'mean',                   # Already per hour, so mean is fine
    'g_use_cum__m3': 'last',           # Last value of the hour
    'heat_gain_ch__J': 'mean',
    'H__W_K_1': 'mean',
    #'temp_demand__degC': 'mean',
    'gas_usage_diff': 'sum'
}

# Resample the DataFrame to hourly frequency and apply the aggregations
df_combined_hourly = df_combined.resample('H').agg(agg_dict)

# Reset the index if you want the timestamp as a column again
df_combined_hourly.reset_index(inplace=True)

# Display the resampled DataFrame
df_combined_hourly


In [ ]:
agg_dict = {
    'id':'last',
    'g_use_ch_inf_cum__kWh': 'last',   # Last value of the hour
    'temp_in__degC': 'mean',         # Average temperature of the hour
    'temp_outside': 'mean',        # Already per hour, so mean is fine
    'GHI': 'mean',                   # Already per hour, so mean is fine
    'g_use_cum__m3': 'last',           # Last value of the hour
    'heat_gain_ch__J': 'mean',
    'H__W_K_1': 'mean',
    #'temp_demand__degC': 'mean',
    'gas_usage_diff': 'sum'
}

# Resample the DataFrame to hourly frequency and apply the aggregations
df_combined_daily = df_combined.resample('D').agg(agg_dict)

# Reset the index if you want the timestamp as a column again
df_combined_daily.reset_index(inplace=True)

# Display the resampled DataFrame
df_combined_daily

In [ ]:
agg_dict = {
    'id':'last',
    'g_use_ch_inf_cum__kWh': 'last',   # Last value of the hour
    'temp_in__degC': 'mean',         # Average temperature of the hour
    'temp_outside': 'mean',        # Already per hour, so mean is fine
    'GHI': 'mean',                   # Already per hour, so mean is fine
    'g_use_cum__m3': 'last',           # Last value of the hour
    'heat_gain_ch__J': 'mean',
    'H__W_K_1': 'mean',
    #'temp_demand__degC': 'mean',
    'gas_usage_diff': 'sum'
}

# Resample the DataFrame to weekly frequency and apply the aggregations
df_combined_weekly = df_combined.resample('W').agg(agg_dict)

# Reset the index if you want the timestamp as a column again
df_combined_weekly.reset_index(inplace=True)

# Display the resampled DataFrame
df_combined_weekly

In [ ]:

df_combined.reset_index(inplace=True)

if 'timestamp' in df_combined.columns:
    df_combined.set_index('timestamp', inplace=True)
else:
    print("Kolom 'timestamp' niet gevonden in df_combined")
df_combined.info()

# Modeling

import seaborn as sns
for house in df_combined_hourly['id'].unique():
    sns.heatmap(df_combined_hourly.corr(), annot=True)
    plt.show()

In [ ]:
for house in df_combined_hourly['id'].unique():
    house_data = df_combined_hourly[df_combined_hourly['id'] == house]
    plt.figure(figsize=(10, 6))
    plt.scatter(df_combined_hourly['gas_usage_diff'], df_combined_hourly['GHI'], label='Gas Usage Diff', alpha=0.5)
    plt.scatter(df_combined_hourly['timestamp'], df_combined_hourly['timestamp'], label='GHI', alpha=0.5)
    plt.xlabel('gas_usage_diff / GHI')
    plt.ylabel('timeline')
    plt.title(f'Scatter Plot for ID: {house}')
    plt.legend()
    plt.show()

In [ ]:
df_combined_hourly.drop(['heat_gain_ch__J','H__W_K_1'], axis=1, inplace=True)
df_combined_hourly

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Constants
air_density = 1.225  # kg/m³, density of air
specific_heat_air = 1005  # J/(kg·K), specific heat capacity of air
house_volume = 250  # m³, estimated volume of the house
floor_area = 100 # m², example floor area
energy_content_per_m3 = 35.8e6  # J/m³ for natural gas

#df['timestamp'] = pd.to_datetime(df_combined['timestamp'])
# Assuming df_combined_hourly is your starting dataframe
df = (df_combined_hourly.copy())
df.reset_index(inplace=True)
# Calculate time differences in seconds
df['time_diff'] = df['timestamp'].diff().dt.total_seconds()#.fillna(0)

# Calculate temperature demand (inside - outside temperature)
df['temp_demand'] = df['temp_in__degC'] - df['temp_outside']

# Calculate the gas usage difference
df['gas_usage_diff'] = df['g_use_cum__m3'].diff().fillna(0)

# Convert gas usage to energy in Joules (assuming natural gas energy content)
df['energy_from_gas'] = df['gas_usage_diff'] * energy_content_per_m3

# Calculate energy required to change the inside temperature
air_mass = air_density * house_volume
df['energy_needed'] = df['temp_demand'] * air_mass * specific_heat_air * (df['time_diff'] / 3600)  # Scaling for hourly data

# Calculate solar energy input
df['solar_energy_input'] = df['GHI'] * df['time_diff'] * 10000  # Convert J/(h·cm²) to J/m²

# Add floor area as a feature
df['floor_area'] = floor_area

# Remove all rows with any NaN values
#df.dropna(inplace=True)

# Calculate the difference in energy due to solar radiation
df['solar_energy_diff'] = df['energy_needed'] - df['energy_from_gas']

# Estimate window area using the solar energy difference
df['estimated_window_area'] = df['solar_energy_diff'] / df['solar_energy_input']

# Drop inf values that might result from division by zero or other issues
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=['estimated_window_area'], inplace=True)

# Drop any rows with remaining NaN values
df.dropna(inplace=True)

In [ ]:
# Define feature columns and target variable
features = ['temp_demand', 'GHI', 'time_diff', 'energy_from_gas', 'floor_area']
target = 'estimated_window_area'

# Split the data into training and test sets
X = df[features]
y = df[target]
(X_train, X_test, y_train, y_test) = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the linear regression model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')

# Function to predict optimal window area
def predict_window_area(temp_demand, GHI, time_diff, energy_from_gas, floor_area):
    data = np.array([[temp_demand, GHI, time_diff, energy_from_gas, floor_area]])
    return model.predict(data)[0]

# Example prediction
temp_demand = 1  # example temperature demand in degrees Celsius
GHI = 200  # example solar irradiance in W/m²
time_diff = 3600  # example time difference in seconds (1 hour)
energy_from_gas = 5000000  # example energy from gas in Joules
floor_area = 100  # example floor area in m²

predicted_area = predict_window_area(temp_demand, GHI, time_diff, energy_from_gas, floor_area)
print(f'Predicted window area: {predicted_area:.2f} m²')